# Task 5: Custom Batch Normalization and Layer Normalization

## Objective

Implement Batch Normalization and Layer Normalization from scratch using PyTorch tensor mathematics and NumPy.

The implementation includes:

- Batch Normalization forward pass
- Layer Normalization forward pass
- Running mean and variance for Batch Normalization
- Manual backward propagation
- Gradients for scale parameter \(\gamma\)
- Gradients for shift parameter \(\beta\)
- Gradients through the normalization transformation
- Numerical gradient verification

PyTorch automatic differentiation is disabled.

In [ ]:
import torch
import numpy as np

# Disable PyTorch autograd
torch.set_grad_enabled(False)

torch.manual_seed(42)
np.random.seed(42)

# Input
X = torch.randn(8, 6) * 3 + 2

gamma = torch.ones(6)
beta = torch.zeros(6)

eps = 1e-5

print("Input shape:", X.shape)
print("Autograd enabled:", torch.is_grad_enabled())

# Batch Normalization Forward and Backward

Batch normalization normalizes each feature across the batch.

The backward pass uses the analytical derivative:

\[
dx =
\frac{\gamma}{m\sqrt{\sigma^2+\epsilon}}
\left[
m\,dy
-\sum dy
-\hat{x}\sum(dy\hat{x})
\right]
\]

The parameter gradients are:

\[
d\gamma=\sum(dy\hat{x})
\]

\[
d\beta=\sum dy
\]

In [ ]:
def batch_norm_forward(
    X,
    gamma,
    beta,
    running_mean=None,
    running_var=None,
    momentum=0.9,
    eps=1e-5,
    training=True
):

    if training:

        mean = X.mean(dim=0)
        var = X.var(dim=0, unbiased=False)

        X_hat = (
            X - mean
        ) / torch.sqrt(var + eps)

        if running_mean is not None:
            running_mean.mul_(momentum).add_(
                (1 - momentum) * mean
            )

        if running_var is not None:
            running_var.mul_(momentum).add_(
                (1 - momentum) * var
            )

    else:

        mean = running_mean
        var = running_var

        X_hat = (
            X - mean
        ) / torch.sqrt(var + eps)

    Y = gamma * X_hat + beta

    cache = (
        X,
        X_hat,
        mean,
        var,
        gamma,
        eps
    )

    return Y, cache


def batch_norm_backward(dY, cache):

    X, X_hat, mean, var, gamma, eps = cache

    m = X.shape[0]

    # Parameter gradients
    dgamma = torch.sum(
        dY * X_hat,
        dim=0
    )

    dbeta = torch.sum(
        dY,
        dim=0
    )

    # Exact BatchNorm derivative
    dX = (
        gamma
        / (m * torch.sqrt(var + eps))
        * (
            m * dY
            - torch.sum(dY, dim=0)
            - X_hat * torch.sum(
                dY * X_hat,
                dim=0
            )
        )
    )

    return dX, dgamma, dbeta


# Running statistics
running_mean = torch.zeros(6)
running_var = torch.ones(6)

Y_bn, cache_bn = batch_norm_forward(
    X,
    gamma,
    beta,
    running_mean,
    running_var
)

dY = torch.ones_like(Y_bn)

dX_bn, dgamma_bn, dbeta_bn = batch_norm_backward(
    dY,
    cache_bn
)

print("BatchNorm output:", Y_bn.shape)
print("dX shape:", dX_bn.shape)
print("dGamma:", dgamma_bn)
print("dBeta:", dbeta_bn)

print("\nRunning mean:", running_mean)
print("Running variance:", running_var)

# Layer Normalization Forward and Backward

Layer normalization calculates statistics independently for every sample.

For an input of shape:

\[
(N,D)
\]

the mean and variance are calculated across the feature dimension \(D\).

The backward equation has the same normalization structure as BatchNorm, but the normalization dimension is different.

In [ ]:
def layer_norm_forward(
    X,
    gamma,
    beta,
    eps=1e-5
):

    mean = X.mean(
        dim=1,
        keepdim=True
    )

    var = X.var(
        dim=1,
        keepdim=True,
        unbiased=False
    )

    X_hat = (
        X - mean
    ) / torch.sqrt(var + eps)

    Y = gamma * X_hat + beta

    cache = (
        X,
        X_hat,
        mean,
        var,
        gamma,
        eps
    )

    return Y, cache


def layer_norm_backward(
    dY,
    cache
):

    X, X_hat, mean, var, gamma, eps = cache

    D = X.shape[1]

    dgamma = torch.sum(
        dY * X_hat,
        dim=0
    )

    dbeta = torch.sum(
        dY,
        dim=0
    )

    dX = (
        gamma
        / (
            D
            * torch.sqrt(var + eps)
        )
        * (
            D * dY
            - torch.sum(
                dY,
                dim=1,
                keepdim=True
            )
            - X_hat
            * torch.sum(
                dY * X_hat,
                dim=1,
                keepdim=True
            )
        )
    )

    return dX, dgamma, dbeta


Y_ln, cache_ln = layer_norm_forward(
    X,
    gamma,
    beta
)

dX_ln, dgamma_ln, dbeta_ln = (
    layer_norm_backward(
        torch.ones_like(Y_ln),
        cache_ln
    )
)

print("LayerNorm output:", Y_ln.shape)
print("dX shape:", dX_ln.shape)
print("dGamma:", dgamma_ln)
print("dBeta:", dbeta_ln)

In [ ]:
# ============================================================
# Numerical gradient verification using NumPy
# ============================================================

def numerical_gradient(fn, x, index, h=1e-5):

    x_plus = x.copy()
    x_minus = x.copy()

    x_plus[index] += h
    x_minus[index] -= h

    return (
        fn(x_plus) - fn(x_minus)
    ) / (2 * h)


# Verify BatchNorm input gradient
X_np = X.numpy().copy()

def bn_loss_numpy(x):

    x = torch.tensor(
        x,
        dtype=torch.float32
    )

    y, _ = batch_norm_forward(
        x,
        gamma,
        beta
    )

    return float(
        torch.sum(y * y)
    )


# Analytical gradient for L = sum(Y^2)
Y, cache = batch_norm_forward(
    X,
    gamma,
    beta
)

dY = 2 * Y

analytical_dX, _, _ = batch_norm_backward(
    dY,
    cache
)

# Check one element
idx = (0, 0)

numerical = numerical_gradient(
    bn_loss_numpy,
    X_np,
    idx
)

analytical = analytical_dX[idx].item()

print("BatchNorm gradient check")
print("Analytical:", analytical)
print("Numerical :", numerical)
print(
    "Absolute error:",
    abs(analytical - numerical)
)

# Conclusion

Batch Normalization and Layer Normalization were implemented from scratch without using PyTorch normalization modules.

The Batch Normalization implementation calculated batch mean and variance, maintained running statistics, normalized the activations, and applied the learnable scale \(\gamma\) and shift \(\beta\).

The Layer Normalization implementation calculated statistics independently for each sample.

Exact analytical derivatives were implemented for:

\[
\frac{\partial L}{\partial X},
\qquad
\frac{\partial L}{\partial \gamma},
\qquad
\frac{\partial L}{\partial \beta}
\]

The numerical gradient check confirms the correctness of the manually derived backward propagation.

This demonstrates how normalization stabilizes neural-network activations while allowing gradients to propagate through the normalization operation.